# Reproduce EGNN Results
This notebook loads the best model checkpoint and its configuration to regenerate metrics on the QM9 test set.

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter
from tqdm.auto import tqdm

device = torch.device('cpu')
print(f"Using device: {device}")

## 1. Model Definition

In [ ]:
ATOM_MAP     = {1: 0, 6: 1, 7: 2, 8: 3, 9: 4}
CHARGE_SCALE = 9.0

def build_node_features(z):
    idx     = torch.zeros_like(z)
    for atomic_num, one_hot_idx in ATOM_MAP.items():
        idx[z == atomic_num] = one_hot_idx
    one_hot = F.one_hot(idx, num_classes=5).float()
    charge_norm   = (z.float() / CHARGE_SCALE).unsqueeze(-1)
    charge_tensor = torch.cat([
        torch.ones_like(charge_norm),
        charge_norm,
        charge_norm ** 2
    ], dim=-1)
    return (one_hot.unsqueeze(-1) * charge_tensor.unsqueeze(1)).view(z.shape[0], -1)


class EGNNLayer(nn.Module):
    def __init__(self, hidden_nf):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_nf * 2 + 1, hidden_nf), nn.SiLU(),
            nn.Linear(hidden_nf, hidden_nf), nn.SiLU(),
        )
        self.att_mlp  = nn.Sequential(nn.Linear(hidden_nf, 1), nn.Sigmoid())
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf * 2, hidden_nf), nn.SiLU(),
            nn.Linear(hidden_nf, hidden_nf),
        )

    def forward(self, h, pos, edge_index):
        row, col  = edge_index
        radial    = ((pos[row] - pos[col]) ** 2).sum(dim=1, keepdim=True)
        edge_feat = self.edge_mlp(torch.cat([h[row], h[col], radial], dim=1))
        edge_feat = edge_feat * self.att_mlp(edge_feat)
        aggr      = scatter(edge_feat, row, dim=0, dim_size=h.size(0), reduce='sum')
        return h + self.node_mlp(torch.cat([h, aggr], dim=1))


class EGNNModel(nn.Module):
    def __init__(self, in_node_nf=15, hidden_nf=128, n_layers=7):
        super().__init__()
        self.embedding = nn.Linear(in_node_nf, hidden_nf)
        self.layers    = nn.ModuleList([EGNNLayer(hidden_nf) for _ in range(n_layers)])
        self.node_dec  = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), nn.SiLU(),
            nn.Linear(hidden_nf, hidden_nf),
        )
        self.graph_dec = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), nn.SiLU(),
            nn.Linear(hidden_nf, 1),
        )

    def forward(self, z, pos, edge_index, batch):
        h = self.embedding(build_node_features(z))
        for layer in self.layers:
            h = layer(h, pos, edge_index)
        h = self.node_dec(h)
        h = scatter(h, batch, dim=0, reduce='sum')
        return self.graph_dec(h).squeeze(-1)


def make_fc_edge_index(batch):
    rows, cols = [], []
    for g in batch.unique():
        mask = (batch == g).nonzero(as_tuple=True)[0]
        n    = mask.size(0)
        rows.append(mask.repeat_interleave(n))
        cols.append(mask.repeat(n))
    return torch.stack([torch.cat(rows), torch.cat(cols)])

## 2. Load Configuration and Initialize Model

In [ ]:
config_path = 'egnn_config.json'
model_path  = 'egnn_best_model.pt'

if not os.path.exists(config_path) or not os.path.exists(model_path):
    raise FileNotFoundError("Required model or config file missing. Please run training first.")

with open(config_path, 'r') as f:
    config = json.load(f)

model = EGNNModel(
    in_node_nf=config['in_node_nf'],
    hidden_nf=config['hidden_nf'],
    n_layers=config['n_layers'],
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model and configuration loaded successfully.")

## 3. Load Dataset

In [ ]:
dataset      = QM9(root='./data/QM9')
test_dataset = dataset[118000:]
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

target_mean = config['target_mean']
target_mad  = config['target_mad']
TARGET_IDX  = config['target_idx']

def denormalize(y_norm):
    return y_norm * target_mad + target_mean

print(f"Test set: {len(test_dataset)} molecules")

## 4. Evaluate and Report

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    total_mae = 0.0
    for batch in tqdm(loader, desc='Evaluating'):
        batch    = batch.to(device)
        fc_edges = make_fc_edge_index(batch.batch).to(device)
        preds    = denormalize(model(batch.z, batch.pos, fc_edges, batch.batch))
        total_mae += (preds - batch.y[:, TARGET_IDX]).abs().sum().item()
    return total_mae / len(loader.dataset)

test_mae = evaluate(model, test_loader)
print(f"Reproduced Test MAE: {test_mae * 1000:.2f} meV")